# Exercises 4 and 5 · Model Card + Datasheet — Project: input guardrail on Aegis (NVIDIA)

**What this notebook does:** it reproduces Exercises 4, 5 and 6 of Dr. Gaur's notebook (Session 7) on **another dataset**: the *Aegis AI Content Safety Dataset 1.0* (`nvidia/Aegis-AI-Content-Safety-Dataset-1.0`), described in Ghosh et al. (2024), *AEGIS: Online Adaptive AI Content Safety Moderation with Ensemble of LLM Experts*, arXiv:2404.05993. It is one of the three datasets used in Moraes et al. (2026), *Safeguarding Big Data Question-Answering Systems with a Two-layer Input Guardrails* (repository `big-qa-architecture`).

**Documented model:** a lightweight *safe / unsafe* classifier (TF-IDF + logistic regression), equivalent to the "fast first layer" of the paper's two-layer guardrail. The binary labels are derived from the annotators' votes with **the same rule the repository uses** (`src/dataset.py`, class `AegisDataset`).

**How to use it (Colab):**
1. Aegis 1.0 is a **gated** dataset: go to https://huggingface.co/datasets/nvidia/Aegis-AI-Content-Safety-Dataset-1.0 with your account and accept the terms.
2. Create a read token at https://huggingface.co/settings/tokens and store it in Colab as a secret named `HF_TOKEN` (🔑 icon in the left sidebar, with access enabled for this notebook).
3. Run the cells in order. No GPU is needed: the full training takes under a minute.

All the numbers in the Model Card (accuracy, macro-F1, F1 of *unsafe*, AUPRC, recall per class and per subgroup) are computed in the run. The Datasheet facts come from the dataset card on Hugging Face and from Ghosh et al. (2024).

> ⚠️ The dataset contains offensive and harmful text (it is *red-teaming* material). This notebook **does not print examples**, only counts and metrics.

In [1]:
# --- Install dependencies and imports ---
%pip -q install -U datasets
import os, platform
import numpy as np, pandas as pd, sklearn
from collections import Counter
from datetime import date
from datasets import load_dataset
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (accuracy_score, f1_score, precision_score, recall_score,
                             average_precision_score)

# --- Constants of the experiment ---
SEEDS = range(5)          # same 5 seeds as in class
TEST_SIZE = 0.20          # 80/20 split of the 'train' split (the official 'test' split is also evaluated)
MIN_N = 20                # minimum subgroup size for reporting disaggregated recall
DATASET_ID = "nvidia/Aegis-AI-Content-Safety-Dataset-1.0"
CLASSES = ["unsafe", "safe"]   # same order as CLASSIFICATION_LABELS in the repo's src/model.py

In [2]:
# --- Hugging Face authentication (gated dataset) ---
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    print("✅ HF_TOKEN loaded from Colab secrets.")
except Exception as e:
    # Outside Colab, or the secret is not configured: interactive login.
    # If you use "Run all", wait for the login to finish before the next cell.
    print("Could not read the HF_TOKEN secret (", type(e).__name__, "): please log in manually.")
    from huggingface_hub import notebook_login
    notebook_login()

✅ HF_TOKEN loaded from Colab secrets.


In [3]:
# --- Load the dataset (Aegis 1.0) ---
raw = load_dataset(DATASET_ID)
train_df = raw["train"].to_pandas()
test_df = raw["test"].to_pandas()
LABEL_COLS = sorted(c for c in train_df.columns if c.startswith("labels_"))
print(raw)
print("Annotation columns:", LABEL_COLS)


def annotations(row):
    """Return the non-empty annotations of a row (one per annotator)."""
    out = []
    for c in LABEL_COLS:
        v = row[c]
        if v is None or (isinstance(v, float) and np.isnan(v)) or str(v).strip() == "":
            continue
        out.append(str(v).strip())
    return out


def repo_label(row):
    """Binary label following the repo's rule (src/dataset.py, AegisDataset):
    'safe' if >= 2 annotations say "Safe"; otherwise 'unsafe'."""
    safe = sum(1 for a in annotations(row) if a.lower() == "safe")
    return "safe" if safe >= 2 else "unsafe"


def majority_label(row):
    """Alternative label: strict majority among the annotations present (tie -> 'unsafe')."""
    ann = annotations(row)
    safe = sum(1 for a in ann if a.lower() == "safe")
    return "safe" if ann and safe > len(ann) / 2 else "unsafe"


def categories(row):
    """Risk categories assigned by at least one annotator.
    One annotation may list several categories separated by commas."""
    cats = set()
    for a in annotations(row):
        for part in a.split(","):
            p = part.strip()
            if p and p.lower() != "safe":
                cats.add(p)
    return sorted(cats)


for df in (train_df, test_df):
    df["text"] = df["text"].fillna("")
    df["n_ann"] = df.apply(lambda r: len(annotations(r)), axis=1)
    df["label"] = df.apply(repo_label, axis=1)
    df["label_majority"] = df.apply(majority_label, axis=1)
    df["categories"] = df.apply(categories, axis=1)

# --- Counts and description of the data (reused later in the Model Card and the Datasheet) ---
N_TR, N_TE = len(train_df), len(test_df)
counts = train_df["label"].value_counts()
class_counts_txt = ", ".join(f"{k}: {v} ({v/N_TR:.1%})" for k, v in counts.items())
test_counts_txt = ", ".join(f"{k}: {v} ({v/N_TE:.1%})" for k, v in test_df["label"].value_counts().items())
ann_dist = {int(k): int(v) for k, v in sorted(Counter(train_df["n_ann"]).items())}
ann_dist_txt = ", ".join(f"{k} annotation(s): {v}" for k, v in ann_dist.items())
has_text_type = "text_type" in train_df.columns
text_type_txt = (", ".join(f"{k}: {v}" for k, v in train_df["text_type"].value_counts().items())
                 if has_text_type else "column 'text_type' not found")
cat_counts = Counter(c for cs in train_df["categories"] for c in cs)
cat_counts_txt = ", ".join(f"{c}: {n}" for c, n in cat_counts.most_common())
rule_disagree = int((train_df["label"] != train_df["label_majority"]).sum())
single_safe = int(((train_df["n_ann"] == 1) & (train_df["label_majority"] == "safe")).sum())
no_cat_unsafe = int(((train_df["label"] == "unsafe") & (train_df["categories"].str.len() == 0)).sum())
empty_text = int((train_df["text"].str.strip() == "").sum())

print("Train / test:", N_TR, N_TE)
print("Distribution (train, repo rule):", class_counts_txt)
print("Distribution (test, repo rule):", test_counts_txt)
print("Annotations per text (train):", ann_dist_txt)
print("Text type (train):", text_type_txt)
print("Risk categories (train, >=1 annotator):", cat_counts_txt)
print("Empty texts (train):", empty_text)

# --- Audit of the repository's labeling rule ---
label_audit_txt = (f"repo rule (>=2 'Safe' votes -> safe) vs strict majority of present annotations disagree on "
                   f"{rule_disagree} of {N_TR} train items ({rule_disagree/N_TR:.1%}); "
                   f"{single_safe} items have a single annotation that says 'Safe' but the repo rule labels them 'unsafe'; "
                   f"{no_cat_unsafe} items are 'unsafe' under the repo rule although no annotator assigned any risk category")
print("Rule audit:", label_audit_txt)

# --- Cross-check against external sources ---
REFERENCE = {"train": 10798, "test": 1199,              # dataset card on Hugging Face (verify)
             "train_safe": 3550, "train_unsafe": 7248,  # 'support' in output/*-aegis.out of the big-qa-architecture repo
             "label_cols_in_repo_comment": 4}           # comment in src/dataset.py: "labels_0..labels_3"
check = lambda a, b: "OK" if a == b else "MISMATCH"
ref_check_txt = (f"train {N_TR} vs card {REFERENCE['train']}: {check(N_TR, REFERENCE['train'])}; "
                 f"test {N_TE} vs card {REFERENCE['test']}: {check(N_TE, REFERENCE['test'])}; "
                 f"safe/unsafe {counts.get('safe', 0)}/{counts.get('unsafe', 0)} vs repo output "
                 f"{REFERENCE['train_safe']}/{REFERENCE['train_unsafe']}: "
                 f"{check((counts.get('safe', 0), counts.get('unsafe', 0)), (REFERENCE['train_safe'], REFERENCE['train_unsafe']))}; "
                 f"annotation columns {len(LABEL_COLS)} vs repo code comment {REFERENCE['label_cols_in_repo_comment']}: "
                 f"{check(len(LABEL_COLS), REFERENCE['label_cols_in_repo_comment'])} (the code itself reads labels_0..labels_4)")
print("Cross-check against sources:", ref_check_txt)

README.md:   0%|          | 0.00/9.40k [00:00<?, ?B/s]

Content Moderation Extracted Annotations(…): reconstructing file:   0%|          |  0.00B / 4.91MB            

Content Moderation Extracted Annotations(…): downloading bytes:           |  0.00B            

Content Moderation Extracted Annotations(…): reconstructing file:   0%|          |  0.00B /  559kB            

Content Moderation Extracted Annotations(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/10798 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1199 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['num_annotations', 'id', 'text', 'text_type', 'labels_0', 'labels_1', 'labels_2', 'labels_3', 'labels_4'],
        num_rows: 10798
    })
    test: Dataset({
        features: ['num_annotations', 'id', 'text', 'text_type', 'labels_0', 'labels_1', 'labels_2', 'labels_3', 'labels_4'],
        num_rows: 1199
    })
})
Annotation columns: ['labels_0', 'labels_1', 'labels_2', 'labels_3', 'labels_4']
Train / test: 10798 1199
Distribution (train, repo rule): unsafe: 7248 (67.1%), safe: 3550 (32.9%)
Distribution (test, repo rule): unsafe: 792 (66.1%), safe: 407 (33.9%)
Annotations per text (train): 2 annotation(s): 12, 3 annotation(s): 9751, 4 annotation(s): 1031, 5 annotation(s): 4
Text type (train): combined: 3147, llm_response: 3131, user_message: 3099, multi_turn: 1421
Risk categories (train, >=1 annotator): Criminal Planning/Confessions: 2482, Needs Caution: 2098, Hate/Identity Hate: 1253, Harassment: 1034, Violence: 958, PII/Privacy: 

In [4]:
# --- Experiment: 5 seeds, stratified split; the TF-IDF vectorizer is fitted ONLY on train ---
def build_model(seed):
    """TF-IDF (1–2-grams) + logistic regression, packed into a single pipeline."""
    return make_pipeline(
        TfidfVectorizer(lowercase=True, ngram_range=(1, 2), min_df=2, max_features=50_000, sublinear_tf=True),
        LogisticRegression(C=1.0, max_iter=2000, random_state=seed),
    )

X_all, y_all = train_df["text"], train_df["label"].values
X_off, y_off = test_df["text"], test_df["label"].values

# Subgroups of the official 'test' split (fixed across all seeds)
unsafe_off = y_off == "unsafe"
cat_masks = {c: unsafe_off & test_df["categories"].apply(lambda cs, c=c: c in cs).values
             for c in sorted({c for cs in test_df["categories"] for c in cs})}
cat_masks["(no category; unsafe only by the repo rule)"] = unsafe_off & (test_df["categories"].str.len() == 0).values
small_groups = {c: int(m.sum()) for c, m in cat_masks.items() if m.sum() < MIN_N}   # too small: not reported
cat_masks = {c: m for c, m in cat_masks.items() if m.sum() >= MIN_N}
tt_masks = ({t: (test_df["text_type"] == t).values for t in test_df["text_type"].unique()}
            if has_text_type else {})

# Result accumulators: *_off = official 'test' split; no suffix = 20% held out from train
res = {k: [] for k in ["acc", "f1m", "f1u", "pu", "ap", "maj",
                       "acc_off", "f1m_off", "f1u_off", "pu_off", "ap_off", "maj_off"]}
recalls, recalls_off = [], []
cat_rec, tt_f1, tt_rec = {}, {}, {}

for s in SEEDS:
    # 1) Stratified 80/20 split (changes with each seed) and training of the model and the baseline
    Xtr, Xte, ytr, yte = train_test_split(X_all, y_all, test_size=TEST_SIZE, stratify=y_all, random_state=s)
    model = build_model(s).fit(Xtr, ytr)
    dummy = DummyClassifier(strategy="most_frequent").fit(Xtr, ytr)   # baseline: always predicts the majority class
    unsafe_col = list(model.classes_).index("unsafe")
    # 2) Overall metrics on the 20% held out and on the official 'test'
    for Xe, ye, sfx, rec_list in [(Xte, yte, "", recalls), (X_off, y_off, "_off", recalls_off)]:
        pred = model.predict(Xe)
        proba = model.predict_proba(Xe)[:, unsafe_col]
        res["acc" + sfx].append(accuracy_score(ye, pred))
        res["f1m" + sfx].append(f1_score(ye, pred, average="macro"))
        res["f1u" + sfx].append(f1_score(ye, pred, pos_label="unsafe"))
        res["pu" + sfx].append(precision_score(ye, pred, pos_label="unsafe"))
        res["ap" + sfx].append(average_precision_score(ye == "unsafe", proba))
        res["maj" + sfx].append(accuracy_score(ye, dummy.predict(Xe)))
        rec_list.append(recall_score(ye, pred, average=None, labels=CLASSES))
    # 3) Disaggregated metrics (official 'test' only): by risk category and by text type
    pred_off = model.predict(X_off)
    for c, m in cat_masks.items():
        cat_rec.setdefault(c, []).append(float((pred_off[m] == "unsafe").mean()))
    for t, m in tt_masks.items():
        tt_f1.setdefault(t, []).append(f1_score(y_off[m], pred_off[m], average="macro"))
        um = m & unsafe_off
        if um.sum() > 0:
            tt_rec.setdefault(t, []).append(float((pred_off[um] == "unsafe").mean()))

# --- Summary: mean ± standard deviation (ddof=1) over the 5 seeds ---
ms = lambda a: f"{np.mean(a):.3f} ± {np.std(a, ddof=1):.3f}"
recalls, recalls_off = np.array(recalls), np.array(recalls_off)
per_class_txt = ", ".join(f"{c}: {ms(recalls[:, i])}" for i, c in enumerate(CLASSES))
per_class_off_txt = ", ".join(f"{c}: {ms(recalls_off[:, i])}" for i, c in enumerate(CLASSES))
cat_txt = "; ".join(f"{c} (n={int(cat_masks[c].sum())}): {ms(r)}"
                    for c, r in sorted(cat_rec.items(), key=lambda kv: np.mean(kv[1])))
small_txt = (f" Categories with fewer than {MIN_N} unsafe test items were not reported: "
             + ", ".join(f"{c} (n={n})" for c, n in small_groups.items()) + ".") if small_groups else ""
tt_txt = ("; ".join(f"{t}: macro-F1 {ms(tt_f1[t])}, unsafe recall {ms(tt_rec[t])} (n={int(tt_masks[t].sum())})"
                    for t in tt_masks if t in tt_rec)
          if has_text_type else "column 'text_type' not found — text-type analysis NOT run.")

print("HELD-OUT 20% of train")
print("  Accuracy        :", ms(res["acc"]), "| majority class:", ms(res["maj"]))
print("  Macro-F1        :", ms(res["f1m"]))
print("  F1 (unsafe)     :", ms(res["f1u"]), "| precision (unsafe):", ms(res["pu"]))
print("  AUPRC (unsafe)  :", ms(res["ap"]))
print("  Per-class recall:", per_class_txt)
print(f"OFFICIAL 'test' SPLIT (n={N_TE})")
print("  Accuracy        :", ms(res["acc_off"]), "| majority class:", ms(res["maj_off"]))
print("  Macro-F1        :", ms(res["f1m_off"]))
print("  F1 (unsafe)     :", ms(res["f1u_off"]), "| precision (unsafe):", ms(res["pu_off"]))
print("  AUPRC (unsafe)  :", ms(res["ap_off"]))
print("  Per-class recall:", per_class_off_txt)
print("Recall of 'unsafe' by risk category (official test, lowest to highest):\n ", cat_txt + small_txt)
print("By text type (official test):", tt_txt)

HELD-OUT 20% of train
  Accuracy        : 0.772 ± 0.005 | majority class: 0.671 ± 0.000
  Macro-F1        : 0.687 ± 0.007
  F1 (unsafe)     : 0.851 ± 0.003 | precision (unsafe): 0.761 ± 0.003
  AUPRC (unsafe)  : 0.920 ± 0.004
  Per-class recall: unsafe: 0.965 ± 0.003, safe: 0.380 ± 0.009
OFFICIAL 'test' SPLIT (n=1199)
  Accuracy        : 0.773 ± 0.004 | majority class: 0.661 ± 0.000
  Macro-F1        : 0.698 ± 0.008
  F1 (unsafe)     : 0.849 ± 0.002 | precision (unsafe): 0.759 ± 0.004
  AUPRC (unsafe)  : 0.912 ± 0.002
  Per-class recall: unsafe: 0.962 ± 0.004, safe: 0.406 ± 0.015
Recall of 'unsafe' by risk category (official test, lowest to highest):
  Needs Caution (n=194): 0.924 ± 0.007; Profanity (n=50): 0.936 ± 0.022; Hate/Identity Hate (n=137): 0.953 ± 0.004; PII/Privacy (n=73): 0.964 ± 0.008; Controlled/Regulated Substances (n=66): 0.970 ± 0.011; Criminal Planning/Confessions (n=298): 0.970 ± 0.004; Violence (n=104): 0.983 ± 0.008; Harassment (n=116): 0.986 ± 0.008; Sexual (n=53)

### External reference: guardrails of the `big-qa-architecture` repository
Results saved in `output/*-aegis.out` (class *unsafe*). **They are not directly comparable**: those models were used without retraining (*zero-shot*) on the full `train` split (n = 10,798), whereas our model is trained on Aegis data and evaluated on the official `test` split. In other words, our model has an *in-distribution* advantage.

In [5]:
# Results of the big-qa-architecture repo on Aegis ('unsafe' class): precision, recall and F1
REPO_AEGIS = pd.DataFrame([
    ("BERT (jackhhao/jailbreak-classifier)",      0.6457, 0.0581, 0.1066),
    ("DistilBERT (madhurjindal/Jailbreak-Detector)", 0.6555, 0.0459, 0.0859),
    ("ELECTRA (idanpers/JailBreakModel)",         0.7122, 0.2475, 0.3674),
    ("Samsung SGuard-JailbreakFilter-2B",         0.9296, 0.0728, 0.1351),
    ("ShieldGemma-2B",                            0.9575, 0.3201, 0.4798),
    ("Llama-Guard-3-1B",                          0.6663, 0.4432, 0.5323),
    ("Chain BERT → Samsung",                      0.7388, 0.2947, 0.4213),
    ("Chain BERT → ShieldGemma",                  0.8166, 0.4952, 0.6165),
    ("Chain BERT → Llama-Guard (paper headline)", 0.6762, 0.5875, 0.6287),
], columns=["model", "precision_unsafe", "recall_unsafe", "f1_unsafe"])
REPO_AEGIS["eval_set"] = "Aegis train, zero-shot (n=10798)"
# Row for this notebook: average of the 5 seeds on the official 'test' split
ours = pd.DataFrame([{
    "model": "THIS NOTEBOOK: TF-IDF + LogReg (mean of 5 seeds)",
    "precision_unsafe": round(np.mean(res["pu_off"]), 4),
    "recall_unsafe": round(np.mean(recalls_off[:, CLASSES.index("unsafe")]), 4),
    "f1_unsafe": round(np.mean(res["f1u_off"]), 4),
    "eval_set": f"Aegis official test, trained on Aegis (n={N_TE})",
}])
comparison = pd.concat([REPO_AEGIS, ours], ignore_index=True).sort_values("f1_unsafe", ascending=False)
best_repo = REPO_AEGIS.loc[REPO_AEGIS["f1_unsafe"].idxmax()]
repo_ref_txt = (f"Best guardrail in the big-qa-architecture repo on Aegis: {best_repo['model']} "
                f"(F1 unsafe {best_repo['f1_unsafe']:.4f}, zero-shot on Aegis train); this model: F1 unsafe "
                f"{ms(res['f1u_off'])} on the official test split. Different evaluation sets — not a like-for-like comparison.")
print(repo_ref_txt)
comparison

Best guardrail in the big-qa-architecture repo on Aegis: Chain BERT → Llama-Guard (paper headline) (F1 unsafe 0.6287, zero-shot on Aegis train); this model: F1 unsafe 0.849 ± 0.002 on the official test split. Different evaluation sets — not a like-for-like comparison.


,model,precision_unsafe,recall_unsafe,f1_unsafe,eval_set
9,THIS NOTEBOOK: TF-IDF + LogReg (mean of 5 seeds),0.7591,0.9619,0.8485,"Aegis official test, trained on Aegis (n=1199)"
8,Chain BERT → Llama-Guard (paper headline),0.6762,0.5875,0.6287,"Aegis train, zero-shot (n=10798)"
7,Chain BERT → ShieldGemma,0.8166,0.4952,0.6165,"Aegis train, zero-shot (n=10798)"
5,Llama-Guard-3-1B,0.6663,0.4432,0.5323,"Aegis train, zero-shot (n=10798)"
4,ShieldGemma-2B,0.9575,0.3201,0.4798,"Aegis train, zero-shot (n=10798)"
6,Chain BERT → Samsung,0.7388,0.2947,0.4213,"Aegis train, zero-shot (n=10798)"
2,ELECTRA (idanpers/JailBreakModel),0.7122,0.2475,0.3674,"Aegis train, zero-shot (n=10798)"
3,Samsung SGuard-JailbreakFilter-2B,0.9296,0.0728,0.1351,"Aegis train, zero-shot (n=10798)"
0,BERT (jackhhao/jailbreak-classifier),0.6457,0.0581,0.1066,"Aegis train, zero-shot (n=10798)"
1,DistilBERT (madhurjindal/Jailbreak-Detector),0.6555,0.0459,0.0859,"Aegis train, zero-shot (n=10798)"


## Exercise 4 · Model Card (9 sections)
Same keys and structure as the instructor's notebook (the scorecard counts exactly 9 sections). The numbers come from the previous cells.

In [6]:
# --- Model Card (9 sections). The numbers come from the previous cells ---
model_card = {
    "Model details": {
        "Name / version": "AegisSafety-TFIDF-LR v0.1",
        "Owner / contact": "FELIX RICHAR MENDOZA PUMACAHUA <felix.mendozap@unmsm.edu.pe>",
        "Date": date.today().isoformat(),
        "Architecture": "TF-IDF (word 1–2-grams, min_df=2, max 50,000 features, sublinear tf) + L2 logistic "
                        "regression (C=1.0); binary output safe/unsafe with 0.5 probability threshold; "
                        f"scikit-learn {sklearn.__version__}, Python {platform.python_version()}",
        "License": "Code: MIT (proposed by the author). Data: CC BY 4.0 per the Hugging Face dataset card (verify); "
                   "access is gated behind NVIDIA's terms.",
    },
    "Intended use": {
        "Primary intended uses": "Course exercise (Research Methods, UNMSM) to practise documentation of a content-safety "
                                 "classifier. Research baseline for the fast first layer of a two-layer input guardrail "
                                 "(Moraes et al., 2026) in Big Data QA systems.",
        "Out-of-scope / misuse": "NOT a production safety filter and NOT for moderating real users or making decisions "
                                 "about people. NOT a jailbreak detector: Aegis labels harmful CONTENT, not jailbreak "
                                 "TECHNIQUES (role-play, obfuscation, prompt injection). English only. A bag-of-words "
                                 "model is trivially evaded by paraphrasing, misspelling or encoding.",
        "Intended users": "Doctoral students in this course; reviewers of this exercise.",
    },
    "Factors": {
        "Relevant groups / conditions": "Risk category of the text (Aegis taxonomy: 13 critical categories plus 'Needs "
                                        "Caution'); text type (user prompt vs LLM response"
                                        + ("" if has_text_type else " — column not found") + "); language (English "
                                        "only); origin (red-team prompts, adversarial by construction); annotator "
                                        "disagreement (variable number of annotations per item). Identity groups "
                                        "mentioned in the text are NOT annotated, so identity-term bias cannot be measured "
                                        "with this dataset (UNKNOWN — to investigate).",
        "Evaluation factors": "Reported overall, per class, per risk category (recall of 'unsafe') and per text type.",
    },
    "Metrics": {
        "Performance measures": "Accuracy, macro-F1 and F1/precision of the 'unsafe' class (classes are imbalanced, so "
                                "accuracy alone misleads); AUPRC of the 'unsafe' probability (threshold-free); "
                                "per-class recall. Recall of 'unsafe' = share of harmful texts that the guardrail stops.",
        "Decision thresholds": "0.5 on the predicted probability (no threshold tuning).",
        "Variation approach": f"Mean ± std (ddof=1) over {len(list(SEEDS))} random seeds; each seed changes the "
                              "stratified 80/20 split of Aegis 'train' and the model's random_state.",
    },
    "Evaluation data": {
        "Datasets": f"(a) {int(TEST_SIZE*100)}% stratified held-out split of Aegis 'train', re-drawn for each seed; "
                    f"(b) the official Aegis 'test' split ({N_TE} items; {test_counts_txt}), fixed across seeds.",
        "Preprocessing": "Binary labels derived from annotator votes with the repo rule (safe if >= 2 'Safe' votes). "
                         "TF-IDF vocabulary fit on the training split only (no leakage into test).",
    },
    "Training data": {
        "Datasets": f"{int((1-TEST_SIZE)*100)}% of Aegis 1.0 'train' ({N_TR} items; classes: {class_counts_txt}).",
        "Known limitations": f"Labels depend on the binarisation rule: {label_audit_txt}. "
                             f"Annotations per item: {ann_dist_txt}. Prompts and LLM responses are mixed in one "
                             f"pool ({text_type_txt}). Risk categories are highly imbalanced ({cat_counts_txt}).",
    },
    "Quantitative analyses": {
        "Overall results": f"Held-out: accuracy={ms(res['acc'])}, macro-F1={ms(res['f1m'])}, F1 unsafe={ms(res['f1u'])}, "
                           f"AUPRC={ms(res['ap'])} (majority-class accuracy for reference: {ms(res['maj'])}). "
                           f"Official test: accuracy={ms(res['acc_off'])}, macro-F1={ms(res['f1m_off'])}, "
                           f"F1 unsafe={ms(res['f1u_off'])}, AUPRC={ms(res['ap_off'])} "
                           f"(majority-class accuracy: {ms(res['maj_off'])}). {repo_ref_txt}",
        "Disaggregated results": f"per-class recall (held-out): {per_class_txt}; per-class recall (official test): "
                                 f"{per_class_off_txt}. Recall of 'unsafe' by risk category (official test): "
                                 f"{cat_txt}.{small_txt} By text type: {tt_txt}.",
    },
    "Ethical considerations": {
        "Risks": "False negatives let harmful requests reach the LLM and the knowledge base; false positives over-block "
                 "legitimate questions and may fall disproportionately on texts that mention identity groups or sensitive "
                 "topics (common in toxicity classifiers; not measured here). The training data contain offensive content. "
                 "A transparent linear model can be probed to find evasions (dual use).",
        "Mitigations": "Educational use only; no deployment; examples are never printed. Any real use would need a "
                       "second, stronger layer (e.g., an LLM guardrail), an identity-term bias audit, adversarial "
                       "testing and human review (UNKNOWN — to investigate).",
    },
    "Caveats and recommendations": {
        "Caveats": "In-distribution evaluation only (same source and taxonomy as training). Results will NOT transfer to "
                   "real QA traffic, other languages or jailbreak-style attacks. No hyper-parameter tuning; the comparison "
                   "with the repo's guardrails uses different evaluation sets.",
        "Recommendations": "Evaluate the repo's guardrails on the same official test split; test out-of-distribution on "
                           "TrustAIRLab in-the-wild jailbreak prompts; check calibration and tune the threshold for the "
                           "first-layer role (high precision); report results under the strict-majority label rule; "
                           "run an identity-term bias audit before any claim of usefulness.",
    },
}

def render_card(card, title):
    """Turn the dictionary (sections -> fields) into Markdown."""
    lines = [f"# {title}", ""]
    for section, fields in card.items():
        lines.append(f"## {section}")
        for k, v in fields.items():
            lines.append(f"- **{k}:** {v}")
        lines.append("")
    return "\n".join(lines)

card_md = render_card(model_card, "Model Card — Aegis Content-Safety Input Guardrail (educational use)")
print(card_md)
with open("model_card.md", "w", encoding="utf-8") as f:
    f.write(card_md)
print("\n✅ Saved model_card.md")

# Model Card — Aegis Content-Safety Input Guardrail (educational use)

## Model details
- **Name / version:** AegisSafety-TFIDF-LR v0.1
- **Owner / contact:** FELIX RICHAR MENDOZA PUMACAHUA <felix.mendozap@unmsm.edu.pe>
- **Date:** 2026-10-30
- **Architecture:** TF-IDF (word 1–2-grams, min_df=2, max 50,000 features, sublinear tf) + L2 logistic regression (C=1.0); binary output safe/unsafe with 0.5 probability threshold; scikit-learn 1.6.1, Python 3.13.18
- **License:** Code: MIT (proposed by the author). Data: CC BY 4.0 per the Hugging Face dataset card (verify); access is gated behind NVIDIA's terms.

## Intended use
- **Primary intended uses:** Course exercise (Research Methods, UNMSM) to practise documentation of a content-safety classifier. Research baseline for the fast first layer of a two-layer input guardrail (Moraes et al., 2026) in Big Data QA systems.
- **Out-of-scope / misuse:** NOT a production safety filter and NOT for moderating real users or making decisions about peopl

## Exercise 5 · Datasheet (7 sections)
What was taken from the dataset card on Hugging Face and from Ghosh et al. (2024) is written as fact and marked *(verify)* when it must be confirmed at the source; what those sources do not say stays as `UNKNOWN — to investigate`. The counts are computed in the run.

In [7]:
# --- Datasheet (7 sections) ---
# Facts taken from the dataset card and Ghosh et al. (2024) are written as facts and marked (verify) when they must be confirmed;
# what those sources do not say stays as "UNKNOWN — to investigate". Counts are computed in the run.
datasheet = {
    "Motivation": {
        "Purpose": "Built to train and evaluate LLM content-safety moderation models against a broad risk taxonomy, "
                   "including the AEGIS ensemble of LLM safety experts (Ghosh et al., 2024).",
        "Created by / funded by": "NVIDIA (Ghosh, Varshney, Galinkin, Parisien — verify author list in arXiv:2404.05993). "
                                  "Funding: NVIDIA internal; grant or external funding UNKNOWN — to investigate.",
    },
    "Composition": {
        "What instances represent": f"One row per text, which is either a user prompt or an LLM response ({text_type_txt}). "
                                    f"Split sizes: {N_TR} train / {N_TE} test. Each row has up to {len(LABEL_COLS)} "
                                    f"annotation columns ({', '.join(LABEL_COLS)}); each annotation is 'Safe' or one or "
                                    f"more risk categories. Annotations per item: {ann_dist_txt}. "
                                    f"Cross-check run: {ref_check_txt}.",
        "Labels": "Annotator-level labels from the Aegis taxonomy (13 critical risk categories plus 'Needs Caution'; "
                  "verify the list on the dataset card). Category frequency in train (assigned by >= 1 annotator): "
                  f"{cat_counts_txt}. Binary label used here = repo rule (safe if >= 2 'Safe' votes): {class_counts_txt} "
                  "(imbalanced). No official binary label is provided by the creators.",
        "Sample vs population": "English prompts taken from Anthropic's HH-RLHF harmlessness red-teaming data plus responses "
                                "generated by an LLM (Mistral-7B-v0.1 per the dataset card — verify). Adversarial by "
                                "construction; NOT representative of real user traffic, other languages or of QA questions "
                                "over enterprise documents.",
        "Missing data": f"{empty_text} empty texts in train. Empty annotation slots are expected because items have a "
                        f"variable number of annotators ({ann_dist_txt}); the reason some items have fewer annotations is "
                        "UNKNOWN — to investigate.",
    },
    "Collection process": {
        "How collected": "Prompts sampled from Anthropic HH-RLHF (harmless-base, red-team conversations); LLM responses "
                         "generated by the creators; every text labelled by trained human annotators following the Aegis "
                         "taxonomy and guidelines (Ghosh et al., 2024 — verify annotator count and QA procedure).",
        "Timeframe": "Released in 2024 together with arXiv:2404.05993 (April 2024). Annotation period UNKNOWN — to investigate.",
        "Consent / ethics": "HH-RLHF prompts were written by crowdworkers hired by Anthropic; their consent terms are those "
                            "of the original collection (UNKNOWN — to investigate). Annotators were exposed to harmful "
                            "content; wellbeing measures are described in the paper (verify). Ethics-committee approval: "
                            "UNKNOWN — to investigate.",
    },
    "Preprocessing / cleaning / labeling": {
        "Steps": "Creators' filtering, de-duplication and PII handling: UNKNOWN — to investigate. In this project: "
                 "(1) binary label derived with the rule from big-qa-architecture src/dataset.py; "
                 f"(2) label-rule audit: {label_audit_txt}; (3) the repo code comment mentions labels_0..labels_3 "
                 "while the code reads labels_0..labels_4; (4) TF-IDF features at train time (fit on train only).",
        "Raw data available": "The per-annotator labels are released (labels_* columns), so other aggregation rules can be "
                              "applied. The full HH-RLHF conversations are public separately (Anthropic, MIT license — verify). "
                              "Annotator guidelines in full: UNKNOWN — to investigate.",
    },
    "Uses": {
        "Appropriate uses": "Training and benchmarking content-safety classifiers and LLM guardrails; research on annotator "
                            "disagreement and label aggregation; teaching of responsible-AI documentation.",
        "Should NOT be used for": "Training models to GENERATE harmful content; claiming jailbreak-detection performance "
                                  "without out-of-distribution tests (labels describe harm categories, not attack "
                                  "techniques); profiling or decisions about real people; generalising to non-English text.",
    },
    "Distribution": {
        "How distributed": "Hugging Face Hub, gated (users must accept NVIDIA's terms): "
                           "https://huggingface.co/datasets/nvidia/Aegis-AI-Content-Safety-Dataset-1.0. "
                           "Dataset paper: Ghosh et al. (2024), arXiv:2404.05993.",
        "License / restrictions": "CC BY 4.0 per the dataset card (verify): attribution required; gated access; the card "
                                  "warns that the data contain offensive content.",
    },
    "Maintenance": {
        "Maintainer": "NVIDIA, via the Hugging Face dataset page; a successor dataset (Aegis 2.0) exists (verify). "
                      "Formal maintenance commitment for v1.0: UNKNOWN — to investigate.",
        "Update plan": "None stated for v1.0 on the dataset page or in the paper (verify).",
    },
}

sheet_md = render_card(datasheet, "Datasheet — NVIDIA Aegis AI Content Safety Dataset 1.0 (educational use)")
print(sheet_md)
with open("datasheet.md", "w", encoding="utf-8") as f:
    f.write(sheet_md)
print("\n✅ Saved datasheet.md")

# Datasheet — NVIDIA Aegis AI Content Safety Dataset 1.0 (educational use)

## Motivation
- **Purpose:** Built to train and evaluate LLM content-safety moderation models against a broad risk taxonomy, including the AEGIS ensemble of LLM safety experts (Ghosh et al., 2024).
- **Created by / funded by:** NVIDIA (Ghosh, Varshney, Galinkin, Parisien — verify author list in arXiv:2404.05993). Funding: NVIDIA internal; grant or external funding UNKNOWN — to investigate.

## Composition
- **What instances represent:** One row per text, which is either a user prompt or an LLM response (combined: 3147, llm_response: 3131, user_message: 3099, multi_turn: 1421). Split sizes: 10798 train / 1199 test. Each row has up to 5 annotation columns (labels_0, labels_1, labels_2, labels_3, labels_4); each annotation is 'Safe' or one or more risk categories. Annotations per item: 2 annotation(s): 12, 3 annotation(s): 9751, 4 annotation(s): 1031, 5 annotation(s): 4. Cross-check run: train 10798 vs card 10798:

## Exercise 6 · Scorecard (self-audit)
Same logic as the instructor's function. The goal is **8/8**.

In [8]:
# Self-audit with 8 checks on the Model Card and the Datasheet (same function as the instructor's notebook)
def score_documentation(model_card, datasheet):
    checks = []
    def add(label, passed, why=""):
        checks.append((label, bool(passed), why))
    def flatten(card):
        parts = []
        for fields in card.values():
            for k, v in fields.items():
                parts.append(str(k).lower()); parts.append(str(v).lower())
        return " ".join(parts)
    flat_mc = flatten(model_card); flat_ds = flatten(datasheet)
    add("Model Card has all 9 sections", len(model_card) == 9)
    add("Datasheet has all 7 sections", len(datasheet) == 7)
    add("Intended MISUSE / out-of-scope stated",
        ("not for" in flat_mc) or ("out-of-scope" in flat_mc) or ("should not" in flat_mc))
    add("Performance reported with variation (± / std / seeds)",
        ("±" in flat_mc) or ("std" in flat_mc) or ("seed" in flat_mc))
    add("Metrics disaggregated (per-class / per-group)",
        ("per-class" in flat_mc) or ("per class" in flat_mc) or ("disaggreg" in flat_mc))
    add("Dataset 'should NOT be used for' stated",
        ("should not be used" in flat_ds) or ("not for" in flat_ds))
    add("Collection process described", len(datasheet["Collection process"]) > 0
        and "how collected" in (" ".join(datasheet["Collection process"].keys()).lower()))
    add("Gaps flagged honestly ('unknown')", ("unknown" in flat_mc) or ("unknown" in flat_ds),
        "Flagging what you don't know beats a confident blank.")
    passed = sum(1 for _, p, _ in checks if p)
    print("DOCUMENTATION SCORECARD"); print("=" * 52)
    for label, p, why in checks:
        print("[" + ("✓" if p else "✗") + "] " + label)
        if why and not p: print("      hint: " + why)
    print("=" * 52); print(f"Score: {passed}/{len(checks)}  ({100*passed/len(checks):.0f}%)")
    return passed, len(checks)

_ = score_documentation(model_card, datasheet)

DOCUMENTATION SCORECARD
[✓] Model Card has all 9 sections
[✓] Datasheet has all 7 sections
[✓] Intended MISUSE / out-of-scope stated
[✓] Performance reported with variation (± / std / seeds)
[✓] Metrics disaggregated (per-class / per-group)
[✓] Dataset 'should NOT be used for' stated
[✓] Collection process described
[✓] Gaps flagged honestly ('unknown')
Score: 8/8  (100%)


In [ ]:
# (Optional) Download the two generated files
try:
    from google.colab import files
    files.download("model_card.md")
    files.download("datasheet.md")
except ImportError:
    print("Outside Colab: the files were left in the working directory.")

## Final checklist before submitting
- [ ] I reviewed the dataset card and Ghosh et al. (2024) and confirmed every item marked *(verify)*: license, authors, origin of prompts and responses, number of annotators, list of categories, annotator wellbeing measures, existence of Aegis 2.0.
- [ ] I reviewed the "Cross-check against sources" output (split sizes, safe/unsafe counts against the repo, annotation columns) and noted any difference.
- [ ] I reviewed the "Rule audit" and decided whether the repo's rule (≥ 2 "Safe" votes) is acceptable or whether I should also report the strict majority.
- [ ] I confirmed that the Model Card explicitly says that Aegis measures **harmful content**, not **jailbreak techniques**.
- [ ] Scorecard 8/8.
- [ ] I uploaded `model_card.md` and `datasheet.md` to GitHub (deliverable 7 folder) and/or to the course platform.

**References:** Mitchell et al. (2019), *Model Cards for Model Reporting*, arXiv:1810.03993 · Gebru et al. (2021), *Datasheets for Datasets*, CACM 64(12) · Ghosh, Varshney, Galinkin and Parisien (2024), *AEGIS: Online Adaptive AI Content Safety Moderation with Ensemble of LLM Experts*, arXiv:2404.05993 · NVIDIA, *Aegis AI Content Safety Dataset 1.0*, Hugging Face · Moraes, Onofre and Aguiar (2026), *Safeguarding Big Data Question-Answering Systems with a Two-Layer Input Guardrails*, ICEIS 2026, DOI 10.5220/0014692300004018 · Bai et al. (2022), *Training a Helpful and Harmless Assistant with RLHF*, arXiv:2204.05862.